In [ ]:
# Parameters
city_key = "ywg"
baseline_feed_id = "2024-12-15"
comparison_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Final Figure Package

This notebook assembles Stephenie's PR2 visual layer from canonical shared outputs. It is the last-pass packaging notebook for the report and dashboard, not a separate analysis path.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from ptn_analysis import TransitContext
from ptn_analysis.analysis import (
    PTN_TIER_COLORS,
    PTN_TIER_ORDER,
    WEB_MERCATOR,
    add_consistent_basemap,
    plot_neighbourhood_base,
)
from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)
from ptn_analysis.context.reporting import collect_summary_stats

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

ctx = TransitContext.from_defaults(feed_id=comparison_feed_id)
serving_db = ctx.serving_db
working_db = ctx.working_db

## 2. Dashboard KPI Table

In [ ]:
# KPI Summary Table — Stephenie

try:
    kpi = collect_summary_stats(
        db_instance=serving_db,
        city_key=city_key,
        feed_id=comparison_feed_id,
    )
    print("KPI raw dict:", kpi)
except Exception as e:
    print("KPI collection failed:", repr(e))
    kpi = {
        "num_stops": None,
        "num_edges": None,
        "neighbourhood_count": None,
        "route_count": None,
        "total_neighbourhood_stops": None,
        "min_stop_density_per_km2": None,
        "median_stop_density_per_km2": None,
        "max_stop_density_per_km2": None,
        "jobs_access_neighbourhood_count": None,
        "mean_jobs_access_score": None,
        "max_jobs_access_score": None,
    }

kpi_table = pd.DataFrame(
    {"Metric": list(kpi.keys()), "Value": list(kpi.values())}
)

kpi_table

## 3. PR2 Summary Panel

In [ ]:
# Figure 1: PR2 Summary Panel — 1x2 layout with KPI + route count as legend text

metrics = serving_db.query("""
    SELECT *
    FROM ywg_route_schedule_metrics
    WHERE feed_id = :fid
""", {"fid": comparison_feed_id})

tables = serving_db.query("SHOW TABLES")["name"].tolist()
tiers = None

if "ywg_route_ptn_tiers" in tables:
    tiers = serving_db.query("""
        SELECT feed_id, route_id, ptn_tier
        FROM ywg_route_ptn_tiers
        WHERE feed_id = :fid
    """, {"fid": comparison_feed_id})

elif "ywg_route_classification_features" in tables:
    cols = serving_db.query(
        "PRAGMA table_info('ywg_route_classification_features')"
    )["name"].tolist()

    tier_col = next((c for c in ["ptn_tier", "tier_label", "route_tier"] if c in cols), None)
    has_feed = "feed_id" in cols

    if tier_col is not None:
        if has_feed:
            tiers = serving_db.query(f"""
                SELECT feed_id, route_id, {tier_col} AS ptn_tier
                FROM ywg_route_classification_features
                WHERE feed_id = :fid
            """, {"fid": comparison_feed_id})
        else:
            tiers = serving_db.query(f"""
                SELECT route_id, {tier_col} AS ptn_tier
                FROM ywg_route_classification_features
            """)

if tiers is not None and not tiers.empty:
    if "feed_id" in tiers.columns and "feed_id" in metrics.columns:
        tier_stats = metrics.merge(tiers, on=["feed_id", "route_id"], how="left")
    else:
        tier_stats = metrics.merge(tiers, on="route_id", how="left")
else:
    tier_stats = metrics.copy()
    tier_stats["ptn_tier"] = "Unknown"

tier_stats["ptn_tier"] = tier_stats["ptn_tier"].fillna("Unknown")

summary = (
    tier_stats.groupby("ptn_tier", dropna=False)
    .agg(
        avg_headway=("mean_headway_minutes", "mean"),
        avg_speed=("scheduled_speed_kmh", "mean"),
        route_count=("route_id", "nunique"),
    )
    .reset_index()
)

summary["sort_order"] = summary["ptn_tier"].apply(
    lambda x: PTN_TIER_ORDER.index(x) if x in PTN_TIER_ORDER else 999
)
summary = summary.sort_values("sort_order")

colors = [PTN_TIER_COLORS.get(t, "#7a7a7a") for t in summary["ptn_tier"]]

# --- 1x2 layout: Headway + Speed, with route count & KPIs as text ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor("white")

# Panel 1: Average headway with route count as bar labels
ax1 = axes[0]
bars1 = ax1.bar(range(len(summary)), summary["avg_headway"], color=colors,
                edgecolor="black", linewidth=0.5)
ax1.set_xticks(range(len(summary)))
ax1.set_xticklabels(summary["ptn_tier"], rotation=40, ha="right", fontsize=8)
ax1.set_title("Average Headway (min)", fontsize=11, weight="bold")
ax1.grid(axis="y", alpha=0.25)
ax1.set_axisbelow(True)
# Add headway value + route count inside each bar
for i, (bar, (_, row)) in enumerate(zip(bars1, summary.iterrows())):
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h * 0.5,
             f"{h:.0f} min\n({int(row['route_count'])} rts)",
             ha="center", va="center", fontsize=7, fontweight="bold", color="white")
ax1.set_ylim(0, summary["avg_headway"].max() * 1.1)

# Panel 2: Average speed
ax2 = axes[1]
bars2 = ax2.bar(range(len(summary)), summary["avg_speed"], color=colors,
                edgecolor="black", linewidth=0.5)
ax2.set_xticks(range(len(summary)))
ax2.set_xticklabels(summary["ptn_tier"], rotation=40, ha="right", fontsize=8)
ax2.set_title("Scheduled Speed (km/h)", fontsize=11, weight="bold")
ax2.grid(axis="y", alpha=0.25)
ax2.set_axisbelow(True)
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h * 0.5,
             f"{h:.1f}", ha="center", va="center", fontsize=8, fontweight="bold", color="white")
ax2.set_ylim(0, summary["avg_speed"].max() * 1.15)

# KPI textbox in top-right of figure
med_density = kpi.get('median_stop_density_per_km2', 0)
med_density_str = f"{med_density:.1f}" if med_density else "NA"
kpi_text = (f"Stops: {kpi.get('num_stops', 'NA'):,}  |  "
            f"Routes: {kpi.get('route_count', 'NA')}  |  "
            f"Neighbourhoods: {kpi.get('neighbourhood_count', 'NA')}  |  "
            f"Median density: {med_density_str}/km\u00b2")

fig.suptitle("PTN Service Summary by Tier", fontsize=13, weight="bold", y=0.99)
fig.text(0.5, 0.93, kpi_text, ha="center", va="top", fontsize=9,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#f0f0f0", edgecolor="#ccc", alpha=0.9))

plt.tight_layout(rect=[0, 0, 1, 0.91])

save_report_figure(
    fig,
    "pr2_summary_panel.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 4. Coverage Cluster Map

In [ ]:
# Figure 2: Coverage Cluster Choropleth Map — Stephenie

density_df = serving_db.query("""
    SELECT neighbourhood, stop_density_per_km2
    FROM ywg_neighbourhood_stop_count_density
    WHERE feed_id = :fid
""", {"fid": comparison_feed_id})

density_df["coverage_cat"] = density_df["stop_density_per_km2"].apply(
    lambda d: "High" if pd.notna(d) and d >= 5 else (
        "Medium" if pd.notna(d) and d >= 1 else "Low"
    )
)

cat_colors = {"High": "#1a9850", "Medium": "#fee08b", "Low": "#d73027"}
cat_order = ["High", "Medium", "Low"]

neigh_gdf = working_db.neighbourhood_gdf().to_crs(WEB_MERCATOR)
gdf_cov = neigh_gdf.merge(density_df, on="neighbourhood", how="left")
gdf_cov["coverage_cat"] = gdf_cov["coverage_cat"].fillna("Low")

fig, ax = plt.subplots(figsize=(10, 8))
for cat in cat_order:
    subset = gdf_cov[gdf_cov["coverage_cat"] == cat]
    subset.plot(ax=ax, color=cat_colors[cat], edgecolor="gray", linewidth=0.3,
                label=f"{cat} ({len(subset)})")

from matplotlib.patches import Patch
legend_patches = [Patch(color=cat_colors[c], label=f"{c} ({len(gdf_cov[gdf_cov['coverage_cat'] == c])})")
                  for c in cat_order]
ax.legend(handles=legend_patches, loc="lower left", fontsize=9, framealpha=0.9)

add_consistent_basemap(ax)
ax.set_title("Neighbourhood Coverage Categories", fontsize=13, weight="bold")
plt.tight_layout()

save_report_figure(
    fig,
    "coverage_cluster_map.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 5. Network Community Map

In [ ]:
# Figure 3: Network Community Map — Stephenie (with basemap + edges + neighbourhood labels)
from shapely.geometry import LineString as _LS
from matplotlib.patches import Patch as _Patch

community_df = serving_db.query("SELECT stop_id, community_id FROM ywg_network_communities")
stops_raw = serving_db.query("SELECT stop_id, stop_name, stop_lat, stop_lon FROM ywg_stops")

df = stops_raw.merge(community_df, on="stop_id", how="inner")

# Build edge GeoDataFrame
try:
    edges_raw = serving_db.query("""
        SELECT from_stop_id, to_stop_id FROM ywg_stop_connection_counts
        WHERE feed_id = :fid
    """, {"fid": comparison_feed_id})
except Exception:
    edges_raw = pd.DataFrame(columns=["from_stop_id", "to_stop_id"])

_coords = dict(zip(stops_raw["stop_id"], zip(stops_raw["stop_lon"], stops_raw["stop_lat"])))
edge_lines = []
for _, er in edges_raw.iterrows():
    c1, c2 = _coords.get(er["from_stop_id"]), _coords.get(er["to_stop_id"])
    edge_lines.append(_LS([c1, c2]) if c1 and c2 else None)
edges_raw = edges_raw.copy()
edges_raw["geometry"] = edge_lines
net_edges = gpd.GeoDataFrame(
    edges_raw.dropna(subset=["geometry"]), geometry="geometry", crs="EPSG:4326"
).to_crs(WEB_MERCATOR)

# Assign community to edges
comm_lookup = dict(zip(df["stop_id"], df["community_id"]))
net_edges["community_id"] = net_edges["from_stop_id"].map(comm_lookup)
net_edges = net_edges.dropna(subset=["community_id"])
net_edges["community_id"] = net_edges["community_id"].astype(int)

# Map stops to neighbourhoods for labeling
try:
    stop_nb = working_db.query("""
        SELECT s.stop_id, n.name AS neighbourhood
        FROM ywg_stops s
        JOIN ywg_neighbourhoods n ON ST_Contains(n.geometry, ST_Point(s.stop_lon, s.stop_lat))
        WHERE s.feed_id = :fid
    """, {"fid": comparison_feed_id})
    comm_nb = df.merge(stop_nb, on='stop_id', how='left')
    dom_nb = (comm_nb.groupby('community_id')['neighbourhood']
              .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else ''))
except Exception:
    dom_nb = pd.Series(dtype=str)

neigh_gdf_s = working_db.neighbourhood_gdf().to_crs(WEB_MERCATOR)

# Distinct, high-contrast colors
distinct_colors = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#42d4f4',
                   '#f032e6', '#fabed4', '#469990', '#dcbeff', '#9A6324',
                   '#800000', '#aaffc3', '#000075', '#a9a9a9', '#ffe119']
comm_sizes = net_edges.groupby("community_id").size().sort_values(ascending=False)
unique_comms = comm_sizes.index.tolist()
comm_colors = {c: distinct_colors[i % len(distinct_colors)] for i, c in enumerate(unique_comms)}

fig, ax = plt.subplots(figsize=(14, 10))
plot_neighbourhood_base(ax, neigh_gdf_s, facecolor="none", edgecolor="#888", linewidth=0.4, alpha=0.3)

for cid in unique_comms:
    sub = net_edges[net_edges["community_id"] == cid]
    sub.plot(ax=ax, color=comm_colors[cid], linewidth=0.4, alpha=0.7)

# Label top 8 communities at centroid with dominant neighbourhood
stops_web = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["stop_lon"], df["stop_lat"]),
    crs="EPSG:4326",
).to_crs(WEB_MERCATOR)

for cid in unique_comms[:8]:
    comm_stops = stops_web[stops_web["community_id"] == cid]
    if comm_stops.empty:
        continue
    cx_pt = comm_stops.geometry.x.mean()
    cy_pt = comm_stops.geometry.y.mean()
    label = dom_nb.get(cid, f"C{cid}")
    if not label:
        label = f"C{cid}"
    ax.annotate(
        label[:18], (cx_pt, cy_pt), fontsize=7, fontweight="bold",
        ha="center", va="center", color=comm_colors[cid],
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=comm_colors[cid], alpha=0.85),
    )

top_c = unique_comms[:10]
legend_patches = [_Patch(color=comm_colors[c],
                         label=f"{dom_nb.get(c, f'C{c}')[:15]} ({comm_sizes[c]} edges)")
                  for c in top_c]
ax.legend(handles=legend_patches, loc="lower left", fontsize=7, framealpha=0.9)

add_consistent_basemap(ax)
ax.set_title("Transit Network Communities", fontsize=13, weight="bold")
plt.tight_layout()

save_report_figure(
    fig,
    "network_communities_pr2.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 5b. Before/After Route Map

In [ ]:
# Figure 4: Before/After PTN Route Comparison — Stephenie
# Uses ctx.working_db for consistency

db = working_db

pre_routes = db.query("""
    SELECT DISTINCT TRIM(CAST(route_short_name AS VARCHAR)) AS route_short_name
    FROM ywg_routes
    WHERE feed_id = :fid
""", {"fid": baseline_feed_id})

post_routes = db.query("""
    SELECT DISTINCT TRIM(CAST(route_short_name AS VARCHAR)) AS route_short_name
    FROM ywg_routes
    WHERE feed_id = :fid
""", {"fid": comparison_feed_id})

pre_set = set(pre_routes["route_short_name"].dropna().astype(str))
post_set = set(post_routes["route_short_name"].dropna().astype(str))

retained = len(pre_set & post_set)
gained = len(post_set - pre_set)
lost = len(pre_set - post_set)

comp_df = pd.DataFrame({
    "Category": ["Retained", "Gained", "Lost"],
    "Count": [retained, gained, lost],
})

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")

colors = ["#4daf4a", "#377eb8", "#e41a1c"]

bars = ax.bar(
    comp_df["Category"],
    comp_df["Count"],
    color=colors,
    edgecolor="black",
    linewidth=0.7,
)

ax.set_title("Before/After PTN Route Comparison", fontsize=13, weight="bold")
ax.set_xlabel("Category", fontsize=11)
ax.set_ylabel("Number of Routes", fontsize=11)
ax.tick_params(labelsize=9)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 0.3,
        f"{int(h)}",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

subtitle = f"Baseline: {baseline_feed_id}   |   Current: {comparison_feed_id}"
ax.text(
    0.5, 1.02, subtitle,
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9,
    color="dimgray",
)

plt.tight_layout()

save_report_figure(
    fig,
    "before_after_routes.png",
    figures_dir=figure_output_directory,
    dpi=dpi,
    enabled=save_figures,
)

plt.close(fig)

## 6. QA Notes

Stephenie should verify that every exported filename matches `ptn_analysis/reporting.py`, that no notebook uses a local palette outside the shared visual helpers, and that the dashboard shows the same metrics as the report tables.